
## 📈 Momentum Screener - HTML Report Generator (Test Mode)
This notebook lets you generate HTML reports for past Fridays and update the GitHub Pages `index.html` to preview how the report site will look.
   

In [1]:
 # Setup paths and imports

import sys, os
from datetime import datetime
import pandas as pd
from dotenv import load_dotenv
# Define key directories

NOTEBOOK_DIR = os.getcwd()

BASE_DIR = os.path.abspath(os.path.join(NOTEBOOK_DIR, ".."))
SRC_DIR = os.path.join(BASE_DIR, "src")


REPORT_DIR = os.path.join(BASE_DIR, "reports")

sys.path.insert(0, SRC_DIR)
os.makedirs(REPORT_DIR, exist_ok=True)
load_dotenv()

True

In [2]:
# Import screener modules
from prices import get_target_dates, download_all_required_price_data
from ranking import get_price_snapshots, compute_returns_and_ranks, store_top10_picks
from report import cache_company_data
from emailer import format_html_email

In [9]:
# patch database to take last week return
import sqlite3
BASE_DIR = os.path.abspath(os.path.join(NOTEBOOK_DIR, ".."))

db_path = os.path.join(BASE_DIR, "data", "market_data.sqlite")

conn = sqlite3.connect(db_path)
cur  = conn.cursor()

# --- safety check: only add it if it isn't there already ---
cur.execute("PRAGMA table_info(top10_picks);")
cols = [row[1] for row in cur.fetchall()]    # row[1] = column name
if "last_week_return" not in cols:
    cur.execute("ALTER TABLE top10_picks ADD COLUMN last_week_return TEXT;")
    conn.commit()

conn.close()


In [3]:

# Function to generate a single HTML report\n",
def generate_html_for_date(friday_str):
    print(f"⏳ Generating report for {friday_str}")

    anchor = pd.Timestamp(friday_str)      
    download_all_required_price_data(today = anchor)
    target_dates = get_target_dates(today=anchor)
    df, resolved = get_price_snapshots(target_dates)
    ranks = compute_returns_and_ranks(df, resolved)
    print(ranks)
    print(ranks.dtypes)
    top10 = store_top10_picks(ranks, run_date=anchor)
    
    if top10.empty:
        print("⚠️ No top 10 results to include.")
        return

    tickers = top10["ticker"].tolist()
    cache_company_data(tickers)
    html_content = format_html_email(top10, report_date=anchor)
    
    output_path = os.path.join(REPORT_DIR, f"momentum_{friday_str}.html")
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(html_content)
    
    print(f"✅ Report written: {output_path}")

In [17]:
anchor = pd.Timestamp("2025-06-13")
target_dates = get_target_dates(today=anchor)
target_dates
df, resolved = get_price_snapshots(target_dates)
ranks = compute_returns_and_ranks(df, resolved)
ranks
#print(ranks.dtypes)
top10 = store_top10_picks(ranks, run_date=anchor)
top10

       current_return last_week_return last_month_return  current_rank  \
ticker                                                                   
PLTR           468.7%            12.7%            475.0%           1.0   
GEV            174.7%             0.7%            149.1%           2.0   
AXON           162.0%            -1.7%            128.4%           3.0   
HWM            105.6%            -2.0%             94.1%           4.0   
VST             92.0%             1.8%             56.4%           5.0   

        last_month_rank  rank_change  
ticker                                
PLTR                1.0          0.0  
GEV                 2.0          0.0  
AXON                3.0          0.0  
HWM                 6.0          2.0  
VST                27.0         22.0  
✅ Stored top 10 picks for 2025-06-13


,ticker,current_return,last_week_return,last_month_return,current_rank,last_month_rank,rank_change,date
0,PLTR,468.7%,12.7%,475.0%,1.0,1.0,0.0,2025-06-13
1,GEV,174.7%,0.7%,149.1%,2.0,2.0,0.0,2025-06-13
2,AXON,162.0%,-1.7%,128.4%,3.0,3.0,0.0,2025-06-13
3,HWM,105.6%,-2.0%,94.1%,4.0,6.0,2.0,2025-06-13
4,VST,92.0%,1.8%,56.4%,5.0,27.0,22.0,2025-06-13
5,DASH,88.3%,0.4%,67.3%,7.0,13.0,6.0,2025-06-13
6,NRG,88.0%,-3.2%,80.0%,8.0,10.0,2.0,2025-06-13
7,NFLX,86.9%,-2.8%,81.7%,9.0,9.0,0.0,2025-06-13
8,PM,79.3%,1.2%,65.4%,11.0,15.0,4.0,2025-06-13
9,GILD,74.2%,1.4%,57.4%,12.0,24.0,12.0,2025-06-13


Thinking through Traceback:
- generate_html_for_date

In [10]:
import sys, importlib
importlib.reload(sys.modules["emailer"])

<module 'emailer' from '/Users/zacseidel/Documents/GitHub/momentum-screener/src/emailer.py'>

In [ ]:
generate_html_for_date("2025-06-13")
generate_html_for_date("2025-06-20")
generate_html_for_date("2025-06-27")
generate_html_for_date("2025-07-04")

⏳ Generating report for 2025-06-13
Skipping 2025-06-12 — already in DB
Skipping SPX for 2025-06-12 — already in DB
Skipping 2025-06-05 — already in DB
Skipping SPX for 2025-06-05 — already in DB
Skipping 2024-06-12 — already in DB
Skipping SPX for 2024-06-12 — already in DB
Skipping 2025-05-12 — already in DB
Skipping SPX for 2025-05-12 — already in DB
Fetching grouped prices for 2024-05-12...
No data for 2024-05-12 — trying previous weekday...
Skipping 2024-05-10 — already in DB
Fetching SPX price for 2024-05-12...
No SPX data for 2024-05-12 — trying previous weekday...
Skipping SPX for 2024-05-10 — already in DB
       current_return last_week_return last_month_return  current_rank  \
ticker                                                                   
PLTR           468.7%            12.7%            475.0%           1.0   
GEV            174.7%             0.7%            149.1%           2.0   
AXON           162.0%            -1.7%            128.4%           3.0   
HWM     

  Preparing metadata (setup.py) ... done
  DEPRECATION: Building 'functools' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'functools'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> [72 lines of output]
      running bdist_wheel
      running build
      running build_py
      creating build
      creating build/lib.macosx-10.9-x86_64-3.9
      copying functools.py -> build/lib.macosx-10.9-x86_64-3.9
      running build_ext
      building '_functools' extension
      creating build/temp.macosx-10.9-x86_64-3.9
      creating build/temp.macosx-10.9-x86_

In [ ]:
# Example: Generate reports for 3 recent Fridays
generate_html_for_date("2025-05-02")
generate_html_for_date("2025-05-09")
generate_html_for_date("2025-05-16")
generate_html_for_date("2025-05-23")


⏳ Generating report for 2025-05-02
Skipping 2025-05-01 — already in DB
Skipping SPX for 2025-05-01 — already in DB
Skipping 2024-05-01 — already in DB
Skipping SPX for 2024-05-01 — already in DB
Skipping 2025-04-01 — already in DB
Skipping SPX for 2025-04-01 — already in DB
Skipping 2024-04-01 — already in DB
Skipping SPX for 2024-04-01 — already in DB
✅ Stored top 10 picks for 2025-05-02
🔍 Processing PLTR (1/10)
  📦 Recent news articles: 2
  ✅ News already fresh — skipping API call.
🔍 Processing NFLX (2/10)
  📦 Recent news articles: 4
  ✅ News already fresh — skipping API call.
🔍 Processing AXON (3/10)
  📦 Recent news articles: 0
  📰 Fetching news...
  ⏳ Sleeping for rate limit...
🔍 Processing VST (4/10)
  📦 Recent news articles: 0
  📰 Fetching news...
  ⏳ Sleeping for rate limit...
🔍 Processing TPR (5/10)
  📦 Recent news articles: 1
  ✅ News already fresh — skipping API call.
🔍 Processing PM (6/10)
  📦 Recent news articles: 0
  📰 Fetching news...
  ⏳ Sleeping for rate limit...
🔍 Proc

https://api.polygon.io/v2/aggs/ticker/AAPL/range/1/day/2025-05-10/2025-05-10?adjusted=true&apiKey=X7QL9PhUOMpuV5SGf7Vnqfa4NPLRsKxl


In [37]:


generate_html_for_date("2024-07-12")
generate_html_for_date("2024-07-19")
generate_html_for_date("2024-07-26")
generate_html_for_date("2024-08-02")
generate_html_for_date("2024-08-09")
generate_html_for_date("2024-08-16")
generate_html_for_date("2024-08-23")
generate_html_for_date("2024-08-30")
generate_html_for_date("2024-09-06")
generate_html_for_date("2024-09-13")
generate_html_for_date("2024-09-20")
generate_html_for_date("2024-09-27")
generate_html_for_date("2024-10-04")
generate_html_for_date("2024-10-11")
generate_html_for_date("2024-10-18")
generate_html_for_date("2024-10-25")
generate_html_for_date("2024-11-01")
generate_html_for_date("2024-11-08")
generate_html_for_date("2024-11-15")
generate_html_for_date("2024-11-22")
generate_html_for_date("2024-11-29")
generate_html_for_date("2024-12-06")
generate_html_for_date("2024-12-13")
generate_html_for_date("2024-12-20")
generate_html_for_date("2024-12-27")        




generate_html_for_date("2025-01-03")
generate_html_for_date("2025-01-10")
generate_html_for_date("2025-01-17")
generate_html_for_date("2025-01-24")
generate_html_for_date("2025-01-31")
generate_html_for_date("2025-02-07")
generate_html_for_date("2025-02-14")
generate_html_for_date("2025-02-21")
generate_html_for_date("2025-02-28")
generate_html_for_date("2025-03-07")
generate_html_for_date("2025-03-14")
generate_html_for_date("2025-03-21")
generate_html_for_date("2025-03-28")
generate_html_for_date("2025-04-04")
generate_html_for_date("2025-04-11")
generate_html_for_date("2025-04-18")
generate_html_for_date("2025-04-25")
generate_html_for_date("2025-05-02")
generate_html_for_date("2025-05-09")
generate_html_for_date("2025-05-16")
generate_html_for_date("2025-05-23")





⏳ Generating report for 2024-07-12
Fetching grouped prices for 2024-07-11...
Stored 10551 rows for 2024-07-11
Fetching SPX price for 2024-07-11...
✅ Stored SPX price for 2024-07-11: $511.39
Fetching grouped prices for 2023-07-11...
Stored 10596 rows for 2023-07-11
Fetching SPX price for 2023-07-11...
✅ Stored SPX price for 2023-07-11: $406.56
Fetching grouped prices for 2024-06-11...
Stored 10519 rows for 2024-06-11
Fetching SPX price for 2024-06-11...
✅ Stored SPX price for 2024-06-11: $493.53
Fetching grouped prices for 2023-06-11...
No data for 2023-06-11 — trying previous weekday...
Fetching grouped prices for 2023-06-09...
Stored 10577 rows for 2023-06-09
Fetching SPX price for 2023-06-11...
No SPX data for 2023-06-11 — trying previous weekday...
Fetching SPX price for 2023-06-09...
✅ Stored SPX price for 2023-06-09: $395.03
✅ Stored top 10 picks for 2024-07-12
       current_return last_month_return  current_rank  last_month_rank  \
ticker                                         

In [13]:
generate_html_for_date("2025-05-30")


⏳ Generating report for 2025-05-30
Fetching grouped prices for 2025-05-29...
Stored 11089 rows for 2025-05-29
Fetching SPX price for 2025-05-29...
✅ Stored SPX price for 2025-05-29: $542.32
Fetching grouped prices for 2024-05-29...
Stored 10507 rows for 2024-05-29
Fetching SPX price for 2024-05-29...
✅ Stored SPX price for 2024-05-29: $483.69
Fetching grouped prices for 2025-04-29...
Stored 11015 rows for 2025-04-29
Fetching SPX price for 2025-04-29...
✅ Stored SPX price for 2025-04-29: $509.49
Fetching grouped prices for 2024-04-29...
Stored 10459 rows for 2024-04-29
Fetching SPX price for 2024-04-29...
✅ Stored SPX price for 2024-04-29: $468.84
✅ Stored top 10 picks for 2025-05-30
🔍 Processing PLTR (1/10)
  📦 Recent news articles: 0
  📰 Fetching news...
  ⏳ Sleeping for rate limit...
🔍 Processing GEV (2/10)
  📦 Recent news articles: 0
  📰 Fetching news...
  ⏳ Sleeping for rate limit...
🔍 Processing AXON (3/10)
  📦 Recent news articles: 0
  📰 Fetching news...
  ⏳ Sleeping for rate lim

In [38]:
!python ../scripts/generate_index.py

✅ Sidebar index written to /Users/zacseidel/Documents/GitHub/momentum-screener/index.html


In [6]:
import sqlite3
from datetime import datetime

In [15]:
DB_PATH = '../data/market_data.sqlite'
DB_PATH

'../data/market_data.sqlite'

In [16]:

with sqlite3.connect(DB_PATH) as conn:
    tables = conn.execute("SELECT name FROM sqlite_master WHERE type='table';").fetchall()

# Show the table names
for table in tables:
    print(table)

('index_allocations',)
('daily_prices',)
('index_constituents',)
('company_metadata',)
('company_news',)
('top10_picks',)


In [21]:
with sqlite3.connect(DB_PATH) as conn:
        voo = pd.read_sql(
            "SELECT * FROM top10_picks",
            conn,
        )
voo


,ticker,current_return,last_month_return,current_rank,last_month_rank,rank_change,date
0,PLTR,425.3%,270.4%,1.0,1.0,0.0,2025-05-02
1,NFLX,105.4%,51.1%,5.0,23.0,18.0,2025-05-02
2,AXON,99.3%,73.8%,6.0,8.0,2.0,2025-05-02
3,VST,78.6%,70.9%,7.0,10.0,3.0,2025-05-02
4,TPR,78.5%,51.5%,8.0,22.0,14.0,2025-05-02
5,PM,77.0%,72.9%,9.0,9.0,0.0,2025-05-02
6,FICO,75.5%,49.0%,10.0,25.0,15.0,2025-05-02
7,WMT,65.5%,48.0%,12.0,28.0,16.0,2025-05-02
8,TTWO,64.8%,40.1%,13.0,42.0,29.0,2025-05-02
9,FTNT,64.0%,42.5%,14.0,37.0,23.0,2025-05-02


In [18]:
voo


date
2023-12-01    421.86
2023-12-08    422.92
2023-12-15    433.09
2023-12-22    435.29
2023-12-29    436.80
               ...  
2025-04-24    502.41
2025-05-01    513.35
2025-05-08    519.34
2025-05-15    542.76
2025-05-16    546.26
Name: close, Length: 74, dtype: float64

In [3]:
import sqlite3
import os

# Update this path if needed
BASE_DIR = os.path.abspath("..")  # Assumes you're in the /notebooks directory
DB_PATH = os.path.join(BASE_DIR, "data", "market_data.sqlite")

with sqlite3.connect(DB_PATH) as conn:
    conn.execute("DROP TABLE IF EXISTS top10_picks;")
    print("✅ Dropped 'top10_picks' table.")


✅ Dropped 'top10_picks' table.


In [27]:
import sqlite3
import pandas as pd
import os

# Adjust this if needed
DB_PATH = os.path.join("../data", "market_data.sqlite")

with sqlite3.connect(DB_PATH) as conn:
    voo_dates = pd.read_sql(
        """
        SELECT date, close
        FROM daily_prices
        WHERE ticker = 'PLTR'
        ORDER BY date DESC
        limit 1
        """,
        conn
    )


voo_dates["close"]

0    129.52
Name: close, dtype: float64

In [37]:
import sqlite3
import pandas as pd

DB_PATH = "../data/market_data.sqlite"  # adjust if needed
date_to_check = "2025-05-01"

with sqlite3.connect(DB_PATH) as conn:
    df = pd.read_sql(
        "SELECT ticker, close FROM daily_prices WHERE ticker = 'FICO'",
        conn, params=[date_to_check]
    )

print("🔍 Sample of stored daily_prices for", date_to_check)
print(df.head(25))

# Check types
print("\n🧪 Ticker column types:", df["ticker"].apply(type).unique())
print("🧪 Close column types:", df["close"].apply(type).unique())

# Show all tickers that match the current top 10
top10 = ["PLTR", "NFLX", "AXON", "VST", "TPR", "PM", "FICO", "WMT", "TTWO", "FTNT"]
missing = [t for t in top10 if t not in df["ticker"].str.upper().unique()]
print("\n🚫 Missing tickers from database for that date:", missing)


DatabaseError: Execution failed on sql 'SELECT ticker, close FROM daily_prices WHERE ticker = 'FICO'': Incorrect number of bindings supplied. The current statement uses 0, and there are 1 supplied.

In [5]:
import sqlite3

db_path = "../data/market_data.sqlite"
table_name = "top10_picks"          # e.g. "daily_prices"


with sqlite3.connect(db_path) as conn:
    cur = conn.execute(f"PRAGMA table_info({table_name});")
    schema_rows = cur.fetchall()

# `schema_rows` is a list of tuples:
# (cid, name, type, notnull, dflt_value, pk)
for cid, name, col_type, notnull, default, pk in schema_rows:
    nn = "NOT NULL" if notnull else ""
    pk_flag = "PRIMARY KEY" if pk else ""
    default_str = f"DEFAULT {default}" if default is not None else ""
    print(f"{name:20} {col_type:12} {nn} {default_str} {pk_flag}")


ticker               TEXT           
current_return       TEXT           
last_month_return    TEXT           
current_rank         REAL           
last_month_rank      REAL           
rank_change          REAL           
date                 TEXT           
last_week_return     REAL           


In [18]:
import sqlite3

In [20]:
db_path = "../data/market_data.sqlite"

def q(sql, params=()):
    """Run a quick SELECT and return a DataFrame."""
    with sqlite3.connect(f'file:{db_path}?mode=ro', uri=True) as conn:
        return pd.read_sql(sql, conn, params=params)

anchor = "2025-06-27"          # ← replace with the date you’re generating
q("""
    SELECT MIN(date), MAX(date)
    FROM daily_prices
    WHERE ticker IN ('AVGO','FFIV','PAYC');
""")



,MIN(date),MAX(date)
0,2023-06-09,2025-06-26


In [ ]:
# 📈 1-cell smoke test for chart_module ---------------------------------------
# If chart_module.py is in the same folder as the notebook, the import will work.
# Otherwise, add its directory to sys.path.

import os
from chart_module import plot_stock_chart
import matplotlib.pyplot as plt

# ── 1. Set your Polygon API key (skip if it’s already in your shell env) ─────
# ── 2. Generate the figure ───────────────────────────────────────────────────
fig, (ax_candle, ax_vol) = plot_stock_chart(
    ticker="AAPL",         # <— any valid equity ticker
    save_path="exampleplot.png",        # omit to display inline
    index_ticker="VOO"     # comparison ETF
)

plt.show()  # not strictly needed in Jupyter ≥ IPython 5, but explicit is nice


/var/folders/cv/nmfv_60118b7v59r0zjz8y_40000gn/T/ipykernel_59598/3214341513.py:17: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()  # not strictly needed in Jupyter ≥ IPython 5, but explicit is nice
